In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layrs
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


df = pd.read_csv('KDDTrain+.csv')
df = pd.get_dummies(df)

normal_df = df[df['Class']=='normal']
attack_df = df[df['Class']=='attack']

normal_features = normal_df.drop('Class',axis=1) 
attack_features = attack_df.drop('Class',axis=1)

scaler = StandardScaler()
normal_scaled = scaler.fit_transform(normal_features)
attack_scaled = scaler.transform(attack_features)

X_train, X_test = train_test_split(normal_scaled, test_size=0.2, random_state=42)

input_dim = X_train.shape[1]

model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(input_dim,)),
    layers.Dense(8, activation='relu'),

    layers.Dense(16, activation='relu'),
    layers.Dense(input_dim, activation='linear')
])

model.compile(optimizer='adam',loss='mse',metrics=['mae'])
H = model.fit(X_train, X_train, epochs=10, validation_data=(X_test,X_test))


X_test_preds = model.predict(X_test)
normal_mse = np.mean(np.power(X_test-X_test_preds,2), axis=1)
threshold = np.percentile(normal_mse, 99)

attack_reconstruction = model.predict(attack_scaled)
attack_mse = np.mean(np.power(attack_scaled-attack_reconstruction,2),axis=1)

anomalies = attack_mse > threshold

print(f"Total Detected Intrusions out of {len(attack_scaled)} actual attacks: {sum(anomalies)}")

